In [1]:
import kagglehub
import os 
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report, accuracy_score,confusion_matrix
from sklearn.preprocessing import PowerTransformer
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import MinMaxScaler
from sklearn.naive_bayes import ComplementNB
import glob
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
OUTPUT_DIR = 'data/artifacts'
# Download latest version
path = kagglehub.dataset_download("chethuhn/network-intrusion-dataset")

print("Path to dataset files:", path)

# Danh sách 17 features bạn yêu cầu
selected_features = [
    'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets',
    'Total Length of Fwd Packets', 'Total Length of Bwd Packets',
    'Fwd Packet Length Mean', 'Bwd Packet Length Mean',
    'Flow Bytes/s', 'Flow Packets/s', 'Packet Length Mean',
    'Packet Length Std', 'SYN Flag Count', 'ACK Flag Count',
    'FIN Flag Count', 'RST Flag Count', 'PSH Flag Count', 'URG Flag Count'
]

Path to dataset files: /home/na/.cache/kagglehub/datasets/chethuhn/network-intrusion-dataset/versions/1


In [ ]:
files = os.listdir(path)
print("Các file trong dataset:", files)

# Thông thường dataset này gồm nhiều file CSV, ta sẽ đọc một file hoặc gộp chúng lại
# Giả sử ta đọc file đầu tiên trong danh sách (hoặc file có tên cụ thể)
csv_files = [f for f in files if f.endswith('.csv')]
print("Danh sách file CSV:", csv_files)

# 1. Đọc file CSV (Ví dụ lấy file đầu tiên)

df_list = []
print(f"--- Đang đọc và lọc {len(csv_files)} files ---")
    
for file in csv_files:
    temp_df = pd.read_csv(path + '/' + file)
    temp_df.columns = temp_df.columns.str.strip()
        
    cols_to_keep = selected_features + ['Label']
    cols_present = [col for col in cols_to_keep if col in temp_df.columns]
        
    df_list.append(temp_df[cols_present])

df = pd.concat(df_list, ignore_index=True)
#df = pd.read_csv(file_path)

df.columns = df.columns.str.strip()

target_column = 'Label' 
all_needed_columns = selected_features + [target_column]

# Lọc lấy các cột cần thiết
df_final = df[all_needed_columns].copy()

df_final.replace([np.inf, -np.inf], np.nan, inplace=True)
df_final.dropna(inplace=True)

print("Kích thước dữ liệu sau khi lọc:", df_final.shape)
df_final.head()

datadir=/kaggle/input/datasets/chethuhn/network-intrusion-dataset/
['Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv', 'Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv', 'Tuesday-WorkingHours.pcap_ISCX.csv', 'Wednesday-workingHours.pcap_ISCX.csv', 'Monday-WorkingHours.pcap_ISCX.csv', 'Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv', 'Friday-WorkingHours-Morning.pcap_ISCX.csv', 'Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv']
Tìm thấy 8 file. Bắt đầu gộp dữ liệu...
--- Đang đọc: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
--- Đang đọc: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
--- Đang đọc: Tuesday-WorkingHours.pcap_ISCX.csv
--- Đang đọc: Wednesday-workingHours.pcap_ISCX.csv
--- Đang đọc: Monday-WorkingHours.pcap_ISCX.csv


In [ ]:
# 1. Làm sạch tên cột (dataset này thường bị lỗi có dấu cách thừa ở tên cột)
df.columns = df.columns.str.strip()

# 2. In ra 5 dòng đầu tiên để xem hình thù dữ liệu
print("=== 5 DÒNG DỮ LIỆU ĐẦU TIÊN ===")
display(df.head())

# 3. In ra thông tin tổng quan về các cột (tên cột, số lượng, kiểu dữ liệu)
print("\n=== THÔNG TIN CÁC CỘT DỮ LIỆU ===")
df.info()

# 4. Kiểm tra sự phân bổ của biến mục tiêu (thường là cột 'Label')
print("\n=== PHÂN BỔ CÁC NHÃN TẤN CÔNG (LABEL) ===")
print(df['Label'].value_counts())

In [ ]:
# 2. Lọc dữ liệu chỉ lấy 17 cột đặc trưng này
X_selected = df[selected_features]
y = df['Label']

df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)

X = df.drop(columns=['Label'])
y = df['Label']

#X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import KBinsDiscretizer
from sklearn.naive_bayes import CategoricalNB
from sklearn.metrics import classification_report
import warnings


MAX_SAMPLES = 20000 
df_reduced = df.groupby('Label').apply(lambda x: x.sample(n=min(len(x), MAX_SAMPLES), random_state=42)).reset_index(drop=True)
df_reduced = df.groupby('Label').apply(
    lambda x: x.sample(n=min(len(x), MAX_SAMPLES), random_state=42), 
    include_groups=False
).reset_index(level=0).reset_index(drop=True)


warnings.filterwarnings('ignore') # Tắt các cảnh báo lặt vặt

# 1. Lấy dữ liệu X, y từ df_reduced (tập 20.000 dòng/nhãn đã tạo ở các bước trước)
X = df_reduced[selected_features]
y = df_reduced['Label']

# 2. Chia tập Train/Test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("1. Đang chia giỏ (Binning) các thông số mạng...")
# Chia mỗi cột thành 15 "mức độ" khác nhau (n_bins=15)
# strategy='quantile' giúp ép các giá trị cực đoan (outliers) vào chung 1 giỏ, làm sạch nhiễu
discretizer = KBinsDiscretizer(n_bins=15, encode='ordinal', strategy='quantile')

X_train_binned = discretizer.fit_transform(X_train)
X_test_binned = discretizer.transform(X_test)

print("2. Đang huấn luyện Categorical Naive Bayes...")
# Dùng thuật toán Naive Bayes chuyên trị dữ liệu đã phân loại (Categorical)
cat_nb_model = CategoricalNB()
cat_nb_model.fit(X_train_binned, y_train)

# 3. Dự đoán và đánh giá
y_pred_cat = cat_nb_model.predict(X_test_binned)

print("\n=== KẾT QUẢ CATEGORICAL NAIVE BAYES (VỚI KỸ THUẬT BINNING) ===")
print(classification_report(y_test, y_pred_cat))

# Vẽ và lưu Confusion Matrix (Ma trận nhầm lẫn)
cm = confusion_matrix(y_test, y_pred_cat)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix - Logistic Regression')
plt.ylabel('Thực tế (True Label)')
plt.xlabel('Dự đoán (Predicted Label)')
plt.tight_layout()

#plt.savefig("artifacts/svm_algorithm.png")
os.makedirs(OUTPUT_DIR, exist_ok=True)
plt.savefig(f"{OUTPUT_DIR}/naive_algorithm.png")

print(f"picture save at :{OUTPUT_DIR}/naive_algorithm.png ")

plt.show()

In [ ]:
import joblib
import os

# Tạo thư mục chứa model nếu chưa có
model_path = "data/models"
os.makedirs(model_path, exist_ok=True)

# 1. Lưu mô hình Categorical Naive Bayes
model_filename = f"{model_path}/categorical_nb_model.pkl"
joblib.dump(cat_nb_model, model_filename)

# 2. Lưu bộ tiền xử lý KBinsDiscretizer (Bắt buộc phải có để dùng cho dữ liệu thực tế sau này)
discretizer_filename = f"{model_path}/discretizer.pkl"
joblib.dump(discretizer, discretizer_filename)

print("-" * 30)
print(f"✅ Đã xuất mô hình tại: {model_filename}")
print(f"✅ Đã xuất bộ tiền xử lý tại: {discretizer_filename}")